# Combined Dataset Visualization
This notebook processes and visualizes multiple FLUMY datasets for comparison.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from ipywidgets import interact, IntSlider, Dropdown, Layout, Checkbox, VBox, HBox
import pyvista as pv
from pathlib import Path
import math
import random
import re

In [ ]:
facies_properties = {
    'undefined':           {'val': 0,  'color': "#ff0000", 'grain_size': 0,  'porosity_class': 0,    'description': "Background/Undefined"},
    'channel_lag':         {'val': 1,  'color': "#f1970f", 'grain_size': 13, 'porosity_class': -2,   'description': "Active channel fill, coarse-grained"},
    'point_bar':           {'val': 2,  'color': "#f3dd12", 'grain_size': 10, 'porosity_class': 1,    'description': "Lower energy channel margins"},
    'sand_plug':           {'val': 3,  'color': "#af8f00", 'grain_size': 9,  'porosity_class': 2,    'description': "Fine-grained oxbow/plug fill"},
    'crevasse_splay_core': {'val': 4,  'color': "#fffc65", 'grain_size': 9,  'porosity_class': 2,    'description': "Proximal high-energy splay"},
    'crevasse_channel':    {'val': 5,  'color': "#ffd986", 'grain_size': 8,  'porosity_class': 3,    'description': "Feeder channel for splays"},
    'crevasse_splay_delta':{'val': 6,  'color': "#ff9853", 'grain_size': 7,  'porosity_class': 2,    'description': "Distal fan-like splay deposit"},
    'levee':               {'val': 7,  'color': "#27ae60", 'grain_size': 6,  'porosity_class': 5,    'description': "Sand/silt ridges bordering channel"},
    'overbank':            {'val': 8,  'color': "#33ff00", 'grain_size': 3,  'porosity_class': 8,    'description': "Stabilized/vegetated levee"},
    'mud_plug':            {'val': 9,  'color': "#fff7db", 'grain_size': 2,  'porosity_class': 10.5, 'description': "Fine silts/clays far from channel"},
    'hemipelagic_plug':    {'val': 10, 'color': "#7a7d80", 'grain_size': 2,  'porosity_class': 10.5, 'description': "Silts near active channel belts"},
    'wetland':             {'val': 11, 'color': "#d862f0", 'grain_size': 1,  'porosity_class': 12.5, 'description': "Organic rich, very fine sediment"},
    'draping':             {'val': 12, 'color': "#8dd5e7", 'grain_size': 1,  'porosity_class': 12.5, 'description': "Lateral accretion sand bodies"},
    'pelagic':             {'val': 13, 'color': "#3498db", 'grain_size': 1,  'porosity_class': 12.5, 'description': "Lacustrine clay/silt"}
}

val_to_info = {
    info['val']: {
        'color': info['color'], 
        'name': key.replace('_', ' ').title()
    } 
    for key, info in facies_properties.items()
}

In [ ]:
# Set base directory
base_dir = Path(r"C:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM")
training_dir = base_dir / "datasets" / "training"

# Find available datasets (folders in training_dir)
dataset_paths = [d for d in training_dir.iterdir() if d.is_dir()]
dataset_names = [d.name for d in dataset_paths]

print(f"Found {len(dataset_names)} datasets:")
for i, name in enumerate(dataset_names):
    print(f"{i+1}. {name}")

## Facies Distribution Comparison

In [ ]:
def get_dataset_distribution(dataset_path):
    facies_dir = dataset_path / "samples" / "facies"
    files = list(facies_dir.glob("*.npy"))
    total_counts = np.zeros(14, dtype=np.int64)
    
    print(f"Processing {dataset_path.name} ({len(files)} samples)...")
    for f in files:
        grid = np.load(f)
        counts = np.bincount(grid.ravel(), minlength=14)
        total_counts += counts
        
    total_voxels = total_counts.sum()
    percentages = (total_counts / total_voxels) * 100
    return percentages

all_distributions = []
for path in dataset_paths:
    all_distributions.append(get_dataset_distribution(path))

In [ ]:
def plot_combined_distribution(distributions, dataset_names):
    num_datasets = len(distributions)
    x = np.arange(14)
    width = 0.8 / num_datasets
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for i, dist in enumerate(distributions):
        ax.bar(x + i*width - (num_datasets-1)*width/2, dist, width, label=dataset_names[i])
    
    ax.set_xlabel('Facies Value')
    ax.set_ylabel('Percentage (%)')
    ax.set_title('Facies Distribution Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v}\n{val_to_info[v]['name']}" for v in x], rotation=45, ha='right')
    ax.legend(title="Datasets")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

plot_combined_distribution(all_distributions, dataset_names)

## 2D Side-by-Side Visualization

In [ ]:
def load_sample(dataset_path, sample_name="sample_1.npy"):
    file_path = dataset_path / "samples" / "facies" / sample_name
    if file_path.exists():
        return np.load(file_path)
    return None

def create_side_by_side_viewer(dataset_paths, dataset_names):
    # Load sample 1 for each dataset
    volumes = []
    for p in dataset_paths:
        vol = load_sample(p)
        if vol is not None:
            volumes.append(vol)
    
    if not volumes:
        print("No samples found.")
        return
        
    nz, ny, nx = volumes[0].shape
    
    # Use a fixed colormap for all facies (0-13)
    colors = [val_to_info[v]['color'] for v in range(14)]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(np.arange(15) - 0.5, 14)

    def update_plot(axis, index):
        fig, axes = plt.subplots(1, len(volumes), figsize=(6 * len(volumes), 5), squeeze=False)
        
        for i, (vol, name) in enumerate(zip(volumes, dataset_names)):
            if axis == 'Z (Map View)':
                slc = vol[index, :, :]
                ylabel, xlabel = "y (North)", "x (East)"
            elif axis == 'Y (XS East-West)':
                slc = vol[:, index, :]
                ylabel, xlabel = "z (Depth)", "x (East)"
            else: # X
                slc = vol[:, :, index]
                ylabel, xlabel = "z (Depth)", "y (North)"

            im = axes[0, i].imshow(slc, origin='lower', cmap=cmap, norm=norm, aspect='auto')
            axes[0, i].set_title(f"{name}\n{axis} at {index}")
            axes[0, i].set_ylabel(ylabel)
            axes[0, i].set_xlabel(xlabel)
            
        # Single legend for all subplots
        unique_vals = np.unique([np.unique(v) for v in volumes])
        legend_elements = [mpatches.Patch(facecolor=val_to_info[v]['color'], label=f"({v}) {val_to_info[v]['name']}") 
                           for v in unique_vals if v in val_to_info]
        
        fig.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), title="Facies")
        plt.tight_layout()
        plt.show()

    axis_dropdown = Dropdown(options=['Z (Map View)', 'Y (XS East-West)', 'X (XS North-South)'], value='Z (Map View)', description='Axis:')
    index_slider = IntSlider(min=0, max=nz-1, step=1, value=nz//2, description='Slice:', layout=Layout(width='50%'))

    def on_axis_change(change):
        if change['new'] == 'Z (Map View)':
            index_slider.max = nz - 1
        elif change['new'] == 'Y (XS East-West)':
            index_slider.max = ny - 1
        else:
            index_slider.max = nx - 1
        index_slider.value = index_slider.max // 2

    axis_dropdown.observe(on_axis_change, names='value')
    interact(update_plot, axis=axis_dropdown, index=index_slider)

create_side_by_side_viewer(dataset_paths, dataset_names)

## 3D PyVista Visualization (Off-screen with Legend)

In [ ]:
def plot_3d_combined(dataset_paths, dataset_names):
    num_datasets = len(dataset_paths)
    images = []
    present_facies_all = set()
    
    facies_colors = [val_to_info[v]['color'] for v in range(14)]
    
    for path in dataset_paths:
        vol = load_sample(path)
        if vol is None: 
            images.append(None)
            continue
            
        present_facies_all.update(np.unique(vol).astype(int))
        nz, ny, nx = vol.shape
        
        grid = pv.ImageData()
        grid.dimensions = (nx + 1, ny + 1, nz + 1)
        # Transpose to match PyVista axis ordering (X, Y, Z)
        grid.cell_data['Facies'] = vol.transpose(2, 1, 0).flatten(order='F')
        
        plotter = pv.Plotter(off_screen=True, window_size=(1000, 1000))
        plotter.background_color = 'white'
        
        plotter.add_mesh(
            grid, 
            scalars='Facies',
            cmap=facies_colors, 
            clim=[0, 13], 
            show_edges=False, 
            show_scalar_bar=False, 
            ambient=0.3,
            diffuse=0.7
        )
        plotter.view_isometric()
        
        # Capture the image array
        images.append(plotter.screenshot(None))
        plotter.close()

    # Create Matplotlib Layout
    fig = plt.figure(figsize=(6 * num_datasets + 2, 8))
    gs = gridspec.GridSpec(1, num_datasets + 1, width_ratios=[1]*num_datasets + [0.2], wspace=0.05)
    
    for i, (img, name) in enumerate(zip(images, dataset_names)):
        if img is not None:
            ax = fig.add_subplot(gs[0, i])
            ax.imshow(img)
            ax.set_title(name, fontsize=14, fontweight='bold')
            ax.axis('off')

    # Build the Categorical Legend
    ax_leg = fig.add_subplot(gs[0, -1])
    ax_leg.axis('off') 
    legend_elements = []
    
    for val in sorted(list(present_facies_all)):
        if val in val_to_info:
            patch = mpatches.Patch(
                color=val_to_info[val]['color'], 
                label=f"{val} - {val_to_info[val]['name']}"
            )
            legend_elements.append(patch)

    ax_leg.legend(
        handles=legend_elements, 
        loc='center left', 
        title="Facies Legend",
        title_fontsize=12,
        fontsize=10,
        frameon=False, 
        labelspacing=1.2
    )

    plt.tight_layout()
    plt.show()

plot_3d_combined(dataset_paths, dataset_names)